# 结构化掩码可视化：ReID / MSMT17（TransReID ViT-S + EDEN）
**回答的问题：各训练阶段的 ckpt 当前会怎么蒙。**

口径与其他三臂一致：G = |∂L_task/∂x₀|（task loss = make_loss 原样构造的
ID CE + triplet，与训练逐位一致）-> 划块均值 -> softmax 概率 ->
Gumbel-top-k 现采 -> 蒙后输入 x̃₀。CRN：跨 ckpt 同图同噪声。

ReID 的特点（reid_probe 里已处理）：triplet 是批内配对挖掘，**必须整批
反传**（batch_grad=True）；裸模型 training 分支才返回 (score, feat)，
前向在 eval 下复刻该分支；输入 256×128、mean/std=0.5。
需在 transreid 环境下运行（CenterLoss 构造需 CUDA）。

In [ ]:
import os, sys
import torch

VISUAL = os.path.abspath('.')          # 内核启动时 cwd 就是 visual/
if VISUAL not in sys.path:
    sys.path.insert(0, VISUAL)

import importlib
import probe_core as pc
import reid_probe as rp
importlib.reload(pc)
importlib.reload(rp)
assert getattr(pc, 'MASK_VIZ_VERSION', 0) >= 1, \
    'probe_core 是旧版（无掩码可视化）——请确认 visual/probe_core.py 已更新并重启内核'

In [ ]:
# ---- 配置 ----
import glob, re
YML = os.path.join(rp.REID_DIR, 'configs/msmt17/vit_small_eden_tweight_mask.yml')
CKPTS = sorted(glob.glob(os.path.join(
    rp.REID_DIR, 'log/transreid/msmt17/vit_small_eden_tweight_mask/transformer_eden_*.pth')),
    key=lambda p: int(re.findall(r'\d+', os.path.basename(p))[-1]))
OUT = os.path.join(VISUAL, 'outputs/mask_viz/reid')
NUM_IMAGES = 8
DISPLAY_IMAGES = 4
MASK_RATIO = None     # None = 模型配置值
GUIDED = True         # True = G 引导选址；False = 预热期均匀随机
SEED = 0
os.makedirs(OUT, exist_ok=True)
CKPTS

In [ ]:
# 配置与数据只建一次，所有 ckpt 共用（同图同分布，跨 ckpt 可比）
# num_classes 是建模型与 make_loss 的必需输入，先取数再建模
cfg = rp.build_reid_cfg(YML)
val_loader, num_classes, cam_num, view_num = rp.build_reid_data(cfg)
batch = rp.build_reid_val_batch(val_loader, NUM_IMAGES)

In [ ]:
results, labels = [], []
for ckpt in CKPTS:
    model = rp.build_reid_model(cfg, num_classes, cam_num, view_num, ckpt)
    tag = os.path.splitext(os.path.basename(ckpt))[0]
    # reid 整批一次反传（triplet 批内配对挖掘，单图无配对）：
    # task loss = ID CE + triplet（make_loss 原样构造）
    grad_fn = rp.make_reid_grad_fn(model, cfg, num_classes)
    res = pc.measure_mask_viz(model, batch, grad_fn, batch_grad=True,
                              mask_ratio=MASK_RATIO, guided=GUIDED, seed=SEED)
    pc.save_mask_viz_cache(os.path.join(OUT, f'{tag}_cache.npz'), res)
    fig = pc.plot_mask_viz(res, title=tag, max_images=DISPLAY_IMAGES,
                           save_path=os.path.join(OUT, f'{tag}_mask_viz.png'))
    results.append(res); labels.append(tag)
    print(f'{tag}: masked={res["masked_frac"].mean():.2f} '
          f'(k={res["k"]}/{res["g"]**2}, alpha={res["alpha"]:g})')
    del model; torch.cuda.empty_cache()
fig  # 显示最后一张

In [ ]:
# 跨 ckpt 演化拼图：行 = 训练阶段，列 = 同一批图
for mode in ['masked', 'prob', 'g']:
    pc.plot_mask_viz_evolution(results, labels, mode=mode,
                               max_images=DISPLAY_IMAGES,
                               save_path=os.path.join(OUT, f'evolution_{mode}.png'))
print('saved evolution_{masked,prob,g}.png')

In [ ]:
# 免重测复画：从 npz 缓存重建 res（换 DISPLAY_IMAGES / 标题都行；掩码已在缓存里）
REPLAY_TAG = os.path.splitext(os.path.basename(CKPTS[-1]))[0]  # 改成想复画的 ckpt tag
res = pc.load_mask_viz_cache(os.path.join(OUT, f'{REPLAY_TAG}_cache.npz'))
fig = pc.plot_mask_viz(res, title=f'{REPLAY_TAG} (replay)',
                       max_images=DISPLAY_IMAGES,
                       save_path=os.path.join(OUT, f'{REPLAY_TAG}_replay.png'))
fig